# Integration

In [1]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

import warnings 
warnings.filterwarnings('ignore')

import scipy
from scipy.sparse import csr_matrix as csr

## Set input path

In [3]:
# Set path
base_path = '/stanley/WangLab/Data/Analyzed/2024-12-02-Mingrui-SCZ/'

input_path = os.path.join(base_path, 'filtered h5ad')

out_path = os.path.join(base_path, 'integrated h5ad')
if not os.path.exists(out_path):
    os.mkdir(out_path)

## Generate complete h5ad

In [4]:
rep1_sample = [file for file in os.listdir(input_path) if 'rep1' in file]
rep2_sample = [file for file in os.listdir(input_path) if 'rep2' in file]
all_sample = [file for file in os.listdir(input_path)]

In [5]:
adata_list = []

for folder in all_sample:
    for file in os.listdir(os.path.join(input_path,folder)):
        print(folder,file)
        adata_list.append(sc.read_h5ad(os.path.join(input_path,folder, file)))

In [6]:
adata = ad.concat(adata_list)
adata

In [7]:
sample_to_replicate = {}
for i in range(1,25):
    if i <=12:
        sample_to_replicate[f'sample{i}'] = 'rep1'
    else:
        sample_to_replicate[f'sample{i}'] = 'rep2'

sample_to_replicate

In [9]:
sample_to_coronal_pos = {}
for i in range(1,25):
    if i in [1,2,3,4,21,22,23,24]:
        sample_to_coronal_pos[f'sample{i}'] = 'PFC'
    if i in [5,6,7,8,17,18,19,20]:
        sample_to_coronal_pos[f'sample{i}'] = 'ST'
    if i in [9,10,11,12,13,14,15,16]:
        sample_to_coronal_pos[f'sample{i}'] = 'HP'

sample_to_coronal_pos

In [10]:
replicate_list = []
coronal_pos_list = []

for sample_id in adata.obs['sample']:
    replicate_list.append(sample_to_replicate.get(sample_id, 'Unknown'))
    coronal_pos_list.append(sample_to_coronal_pos.get(sample_id, 'Unknown'))

adata.obs['replicate'] = replicate_list
adata.obs['coronal_position'] = coronal_pos_list

In [11]:
adata.obs['genotype'] = adata.obs['condition']

In [12]:
marker = pd.read_csv('/stanley/WangLab/Data/Analyzed/2024-03-12-Mingrui-PFC/cell type classification/genelist2', header=None)

idx_HVGs = np.isin(np.array([t.capitalize() for t in adata.var.index.values]),np.array([t.capitalize() for t in marker.iloc[:,0]]))
adata.var['highly_variable'] = False
adata.var.loc[idx_HVGs,'highly_variable'] = True

print(np.sum(adata.var['highly_variable']))

In [40]:
adata

## Integration

In [15]:
def preprocess_fast(sdata1, mode='customized', target_sum=1e4, base=2, zero_center=True, regressout=False):
    if type(sdata1.layers['raw']) != scipy.sparse._csr.csr_matrix:
        sdata1.layers['raw'] = csr(sdata1.layers['raw'].copy())
    sdata1.X = sdata1.layers['raw'].copy()
    if mode == 'default':
        sc.pp.normalize_total(sdata1)
        sdata1.layers['norm'] = csr(sdata1.X.copy())
        sc.pp.log1p(sdata1)
        sdata1.layers['log1p_norm'] = csr(sdata1.X.copy())
        sc.pp.scale(sdata1,zero_center = zero_center)
        if scipy.sparse.issparse(sdata1.X): #### automatically change to non csr matrix (zero_center == True, the .X would be sparce)
            sdata1.X = sdata1.X.toarray().copy()
        sdata1.layers['log1p_norm_scaled'] = sdata1.X.copy()
        if regressout:
            sdata1.obs['total_counts'] = sdata1.layers['raw'].toarray().sum(axis=1)
            sc.pp.regress_out(sdata1, ['total_counts'])
            sdata1.layers['log1p_norm_scaled'] = sdata1.X.copy()
        return sdata1 #### sdata1.X is sdata1.layers['log1p_norm_scaled']
    elif mode == 'customized':
        if target_sum == 1e4:
            target_sum_str = '1e4'
        else:
            target_sum_str = str(target_sum)
        sc.pp.normalize_total(sdata1,target_sum=target_sum)
        sdata1.layers[f'norm{target_sum_str}'] = csr(sdata1.X.copy())
        sc.pp.log1p(sdata1,base = base)
        sdata1.layers[f'log{str(base)}_norm{target_sum_str}'] = csr(sdata1.X.copy())
        sc.pp.scale(sdata1,zero_center = zero_center)
        if scipy.sparse.issparse(sdata1.X): #### automatically change to non csr matrix (zero_center == True, the .X would be sparce)
            sdata1.X = sdata1.X.toarray().copy()
        sdata1.layers[f'log{str(base)}_norm{target_sum_str}_scaled'] = sdata1.X.copy()
        if regressout:
            sdata1.obs['total_counts'] = sdata1.layers['raw'].toarray().sum(axis=1)
            sc.pp.regress_out(sdata1, ['total_counts'])
            sdata1.layers[f'log{str(base)}_norm{target_sum_str}_scaled'] = sdata1.X.copy()
        return sdata1 #### sdata1.X is sdata1.layers[f'log{str(base)}_norm{target_sum_str}_scaled']
    else:
        print('Please set the `mode` as one of the {"default", "customized"}.')

def combat_Harmony_integration(
    sdata_inte,
    batch_key_combat,
    batch_key_harmony,
    umap_n_neighbors,
    umap_n_pcs,
    min_dist,
    spread_t,
    scaled_layer = 'log2_norm1e4_scaled',
    use_highly_variable_t = True,
    n_components = 50,
    ifpca = True,
    ifcombat = True,
    ifharmony = True,
    mode = 'rsc'):

    #### integration based on the Harmony
    sdata_inte.X = sdata_inte.layers[scaled_layer].copy()
    if ifcombat:
        print(f'Running combat integration:')
        sc.pp.combat(sdata_inte, key=batch_key_combat)
    if ifpca:
        print(f'Running PCA based on the layer {scaled_layer}:')
        sc.tl.pca(sdata_inte, use_highly_variable=use_highly_variable_t, svd_solver='full', n_comps=n_components) 
    if ifharmony:
        print(f'Running Harmony integration:')
        sc.external.pp.harmony_integrate(sdata_inte, batch_key_harmony)
        pc_feature = 'X_pca_harmony'
    else:
        pc_feature = 'X_pca'
    print(f'Compute a neighborhood graph based on the {umap_n_neighbors} `n_neighbors`, {umap_n_pcs} `n_pcs`:')
    sc.pp.neighbors(sdata_inte, n_neighbors=umap_n_neighbors, n_pcs=umap_n_pcs, use_rep=pc_feature)
    print(f'Generate the UMAP based on the {min_dist} `min_dist`, {spread_t} `spread`:')
    sc.tl.umap(sdata_inte,min_dist=min_dist, spread = spread_t)
    return sdata_inte

In [16]:
adata.obs['protocol_replicate'] = adata.obs['protocol'].astype(str) + '_' + adata.obs['replicate'].astype(str)
adata.obs['genotype_protocol_replicate'] = adata.obs['genotype'].astype(str) + '_' + adata.obs['protocol'].astype(str) + '_' + adata.obs['replicate'].astype(str)
adata.obs['genotype_protocol_replicate'].unique()

In [19]:
preprocess_fast(adata, regressout=True)

adata_integrated = combat_Harmony_integration(
        sdata_inte=adata,
        batch_key_combat='genotype_protocol_replicate',
        batch_key_harmony='genotype_protocol_replicate',
        umap_n_neighbors=50,
        umap_n_pcs=30,
        min_dist=.0001,
        spread_t=5,
        n_components=50,
        ifcombat=True)

from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')

adata_integrated.write_h5ad(f"{out_path}/{date}-all-sample-combined-harmony-2-genotype-protocol-replicate.h5ad")

In [21]:
sc.pl.umap(adata_integrated, color='sample')
sc.pl.umap(adata_integrated, color='total_counts')
sc.pl.umap(adata_integrated, color='n_genes')
sc.pl.umap(adata_integrated, color='replicate')
sc.pl.umap(adata_integrated, color='protocol')

sc.pl.umap(adata_integrated[adata_integrated.obs['replicate']=="rep1"], color='replicate')
sc.pl.umap(adata_integrated[adata_integrated.obs['replicate']=="rep2"], color='replicate')

sc.pl.umap(adata_integrated[adata_integrated.obs['protocol']=="STAR"], color='protocol')
sc.pl.umap(adata_integrated[adata_integrated.obs['protocol']=="RIBO"], color='protocol')

sc.pl.umap(adata_integrated[adata_integrated.obs['sample']=="sample5"], color='sample')